# 🔁 Main Pipeline – Versi Notebook

Notebook ini merupakan versi interaktif dari `main.py`.
Pipeline dijalankan end-to-end untuk analisis sentimen pernyataan Prabowo.

---

**Urutan Pipeline:**
1. 📊 EDA – Exploratory Data Analysis
2. 🧹 Preprocessing – Data Cleaning pada Data Raw
3. 🔢 Encoding TF-IDF – Load `dataset_clean_manual.csv` + TF-IDF
4. 🤖 Modeling – LR, SVM, NB (Sebelum & Sesudah SMOTE)
5. ☁️ Visualisasi WordCloud
6. 📈 Evaluasi Model

In [ ]:
# ─── Setup Path ────────────────────────────────────────────────
# Tambahkan folder src/ agar bisa import modul
import sys
import os

# Jika notebook dijalankan dari folder src/, pakai path saat ini
# Jika dijalankan dari root project, tambahkan src/
NOTEBOOK_DIR = os.path.abspath('')
SRC_DIR = NOTEBOOK_DIR if os.path.basename(NOTEBOOK_DIR) == 'src' else os.path.join(NOTEBOOK_DIR, 'src')
ROOT_DIR = os.path.dirname(SRC_DIR)
OUTPUT_DIR = os.path.join(ROOT_DIR, 'output')

if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Root dir : {ROOT_DIR}')
print(f'Src dir  : {SRC_DIR}')
print(f'Output   : {OUTPUT_DIR}')

In [ ]:
# ─── Import Modul Pipeline ─────────────────────────────────────
from data_loader import load_raw_data, load_processed_data, load_kamus, save_processed_data
from preprocessing import (
    setup_stemmer, load_stopwords_github,
    build_stopword_final, build_kamus_gabungan, run_preprocessing
)
from model import (
    encode_labels, tfidf_vectorize,
    split_data_stratified, apply_smote,
    train_all_models, save_models
)
from utils import run_eda, plot_wordcloud_all_models, evaluate_all_models

print('Semua modul berhasil diimport.')

---
## 📊 TAHAP 1: Exploratory Data Analysis (EDA)

In [ ]:
# Load dataset berlabel untuk EDA
df_labeled = load_processed_data('dataset_clean_manual.csv')
df_labeled.head()

In [ ]:
# Jalankan EDA: statistik + distribusi label (pie/donut chart)
run_eda(df_labeled, label_col='label_manual', save_dir=OUTPUT_DIR)

---
## 🧹 TAHAP 2: Preprocessing – Data Cleaning

In [ ]:
# 2a. Load data raw
RAW_PATH = os.path.join(ROOT_DIR, 'data', 'raw')

df_raw      = load_raw_data('DatasetSentimen.csv')
kamus_excel = load_kamus('kamuskatabaku.xlsx')

print(f'\nDataset raw: {len(df_raw):,} baris')
df_raw.head()

In [ ]:
# 2b. Setup komponen preprocessing
stemmer            = setup_stemmer()
stopword_github, _ = load_stopwords_github()
stopword_final     = build_stopword_final(stopword_github)
kamus_gabungan     = build_kamus_gabungan(kamus_excel)

In [ ]:
# 2c. Jalankan Data Cleaning
# (Proses ini mungkin memakan waktu beberapa menit karena stemming)
df_clean = run_preprocessing(
    df_raw, kamus_gabungan, stopword_final, stemmer, text_col='text'
)
df_clean.head()

In [ ]:
# 2d. Simpan hasil cleaning (opsional)
save_processed_data(df_clean[['text', 'text_clean']], 'dataset_clean_pipeline.csv')

---
## 🔢 TAHAP 3: Encoding TF-IDF

> ⚠️ **Catatan penting:** Tahap ini menggunakan **`dataset_clean_manual.csv`** (hasil labeling manual),
> bukan hasil cleaning dari tahap 2 (yang belum memiliki label).

In [ ]:
# Load dataset berlabel manual
df_modeling = load_processed_data('dataset_clean_manual.csv')

# Standarisasi label
def standarkan_label(label):
    if str(label).strip() == '' or str(label).lower() == 'nan':
        return None
    label = str(label).strip().lower()
    mapping = {
        'positif': 'Positif', 'pos': 'Positif', 'p': 'Positif',
        'negatif': 'Negatif', 'neg': 'Negatif', 'n': 'Negatif',
        'netral':  'Netral',  'net': 'Netral',
    }
    return mapping.get(label, label)

df_modeling['label_manual'] = df_modeling['label_manual'].apply(standarkan_label)
df_modeling = df_modeling[
    df_modeling['label_manual'].isin(['Positif', 'Negatif', 'Netral'])
].reset_index(drop=True)

print(f'Total data: {len(df_modeling):,} baris')
print('\nDistribusi label:')
print(df_modeling['label_manual'].value_counts())
df_modeling.head()

In [ ]:
# Label Encoding
df_modeling, label_encoder = encode_labels(df_modeling, label_col='label_manual')

# Pisahkan fitur dan target
X_text = df_modeling['text_clean']
y      = df_modeling['label_encoded']

print(f'\nJumlah data fitur (X) : {len(X_text):,}')
print(f'Jumlah data target (y): {len(y):,}')

In [ ]:
# TF-IDF Vectorization
X_tfidf, tfidf = tfidf_vectorize(X_text)

---
## 🤖 TAHAP 4: Modeling

Model yang dilatih:
- **Logistic Regression** (sebelum & sesudah SMOTE)
- **Support Vector Machine** (sebelum & sesudah SMOTE)
- **Multinomial Naive Bayes** (sebelum & sesudah SMOTE)

In [ ]:
# Train-Test Split (stratified 80:20)
X_train, X_test, y_train, y_test = split_data_stratified(X_tfidf, y)

In [ ]:
# Terapkan SMOTE pada data latih
X_train_smote, y_train_smote = apply_smote(X_train, y_train)

In [ ]:
# Training semua model (LR, SVM, NB × sebelum/sesudah SMOTE)
models = train_all_models(X_train, y_train, X_train_smote, y_train_smote)

In [ ]:
# Simpan semua model ke folder output/
save_models(models, tfidf, label_encoder, save_dir=OUTPUT_DIR)

---
## ☁️ TAHAP 5: Visualisasi WordCloud

In [ ]:
# Buat WordCloud untuk semua model dan semua kelas sentimen
plot_wordcloud_all_models(models, tfidf, label_encoder, save_dir=OUTPUT_DIR)

---
## 📈 TAHAP 6: Evaluasi Model

In [ ]:
# Evaluasi semua model: Accuracy, Precision, Recall, F1, Confusion Matrix
df_hasil = evaluate_all_models(models, X_test, y_test, label_encoder, save_dir=OUTPUT_DIR)

# Tampilkan tabel ringkasan
df_hasil

---
## ✅ Pipeline Selesai!

Semua output (grafik, model, confusion matrix) tersimpan di folder `output/`.